<center>Universidade Univille</center>
<center>Unidade 04 - Aprendizado de Máquinas</center>
<center>Classificação de Imagens</center>
<center>Ana Paula de Souza</center>
<center>TURMA 2026/2</center>

**<center>TRABALHO: Contador de Carros com Visão Computacional</center>**

---

## Objetivo

Contar automaticamente os carros que passam por uma rua usando OpenCV.

**Técnica utilizada:** Subtração de Fundo (`MOG2`) + Detecção de Contornos + Linha de Contagem Virtual com Rastreamento por ID

**Pipeline:**
1. Carrega um vídeo de tráfego
2. Aplica subtração de fundo para isolar objetos em movimento
3. Detecta contornos dos veículos
4. Rastreia cada veículo por ID entre frames
5. Conta cada veículo que cruza a linha horizontal
6. Exibe o resultado frame a frame e salva o vídeo processado

## 01. Importação de Bibliotecas

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
from IPython.display import display, HTML
import os

print('OpenCV versão:', cv2.__version__)
print('NumPy versão:', np.__version__)

## 02. Carregamento do Vídeo

O vídeo `video.mp4` já foi colocado na mesma pasta do notebook (ou no `/content/` do Colab após o upload).

> **No Google Colab:** clique no ícone de pasta no painel esquerdo, arraste o arquivo `video.mp4` e aguarde o upload terminar antes de continuar.

In [ ]:
ARQUIVO_VIDEO = 'video.mp4'

if os.path.exists(ARQUIVO_VIDEO):
    tamanho = os.path.getsize(ARQUIVO_VIDEO) / (1024 * 1024)
    print(f'Vídeo encontrado: {ARQUIVO_VIDEO} ({tamanho:.1f} MB)')
else:
    print(f'ATENÇÃO: arquivo "{ARQUIVO_VIDEO}" não encontrado.')
    print('Faça o upload do vídeo para a pasta /content/ no Colab.')

## 03. Exploração do Vídeo — Informações Básicas

Antes de processar, vamos inspecionar as propriedades do vídeo e visualizar alguns frames.

In [ ]:
cap = cv2.VideoCapture(ARQUIVO_VIDEO)

largura      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps          = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duracao      = total_frames / fps if fps > 0 else 0

print('=== Informações do Vídeo ===')
print(f'Resolução   : {largura} x {altura} pixels')
print(f'FPS         : {fps:.1f}')
print(f'Total frames: {total_frames}')
print(f'Duração     : {duracao:.1f} segundos')

cap.release()

In [ ]:
# Visualiza 4 frames do vídeo para entender o conteúdo
cap = cv2.VideoCapture(ARQUIVO_VIDEO)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

posicoes = [0, total//4, total//2, 3*total//4]
fig, eixos = plt.subplots(1, 4, figsize=(20, 4))

for i, pos in enumerate(posicoes):
    cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
    ret, frame = cap.read()
    if ret:
        eixos[i].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        eixos[i].set_title(f'Frame {pos}')
        eixos[i].axis('off')

plt.suptitle('Amostras do Vídeo de Tráfego', fontsize=14)
plt.tight_layout()
plt.show()

cap.release()

## 04. Pré-processamento — Conversão de Espaços de Cores

Antes de aplicar a subtração de fundo, visualizamos como o frame aparece em diferentes espaços de cores.

In [ ]:
cap = cv2.VideoCapture(ARQUIVO_VIDEO)
cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) // 2)
ret, frame_amostra = cap.read()
cap.release()

rgb  = cv2.cvtColor(frame_amostra, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(frame_amostra, cv2.COLOR_BGR2GRAY)
hsv  = cv2.cvtColor(frame_amostra, cv2.COLOR_BGR2HSV)
lab  = cv2.cvtColor(frame_amostra, cv2.COLOR_BGR2LAB)

fig, eixos = plt.subplots(2, 2, figsize=(14, 8))

eixos[0,0].imshow(rgb);               eixos[0,0].set_title('RGB');        eixos[0,0].axis('off')
eixos[0,1].imshow(gray, cmap='gray'); eixos[0,1].set_title('Grayscale'); eixos[0,1].axis('off')
eixos[1,0].imshow(hsv);               eixos[1,0].set_title('HSV');        eixos[1,0].axis('off')
eixos[1,1].imshow(lab);               eixos[1,1].set_title('LAB');        eixos[1,1].axis('off')

plt.suptitle('Espaços de Cores — Frame Amostra', fontsize=14)
plt.tight_layout()
plt.show()

print('Dimensões do frame (linhas, colunas, bandas):', frame_amostra.shape)
print('Tamanho total em pixels:', frame_amostra.size)

## 05. Técnica de Detecção — Subtração de Fundo (MOG2)

O algoritmo **MOG2** (Mixture of Gaussians) aprende o "fundo" estático do vídeo e destaca tudo que se move (os carros).

Abaixo comparamos o frame original com a máscara gerada pelo algoritmo.

In [ ]:
subtrator = cv2.createBackgroundSubtractorMOG2(
    history=300, varThreshold=60, detectShadows=True
)

cap = cv2.VideoCapture(ARQUIVO_VIDEO)
exemplos_mascara = []
exemplos_frame   = []
frame_idx = 0
capturas_desejadas = {50, 100, 150, 200}

while True:
    ret, frame = cap.read()
    if not ret:
        break
    mascara = subtrator.apply(frame)
    _, mascara = cv2.threshold(mascara, 200, 255, cv2.THRESH_BINARY)
    if frame_idx in capturas_desejadas:
        exemplos_frame.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        exemplos_mascara.append(mascara.copy())
    frame_idx += 1

cap.release()

n = len(exemplos_frame)
fig, eixos = plt.subplots(2, n, figsize=(5*n, 7))
for i in range(n):
    eixos[0, i].imshow(exemplos_frame[i]);             eixos[0, i].set_title('Frame original'); eixos[0, i].axis('off')
    eixos[1, i].imshow(exemplos_mascara[i], cmap='gray'); eixos[1, i].set_title('Máscara MOG2');   eixos[1, i].axis('off')

plt.suptitle('Subtração de Fundo — Objetos em Movimento Destacados', fontsize=13)
plt.tight_layout()
plt.show()

## 06. Contador de Carros — Algoritmo Principal

**Lógica de contagem:**
- Uma linha horizontal amarela é desenhada no centro do vídeo
- Cada carro recebe um ID único e é rastreado entre frames pelo seu centroide
- Quando o centro de um carro cruza a linha, o contador incrementa
- Um cooldown por ID evita contar o mesmo carro mais de uma vez

In [ ]:
# ─────────────────────────────────────────────────
# PARÂMETROS — ajuste conforme o seu vídeo
# ─────────────────────────────────────────────────
AREA_MINIMA     = 1500  # área mínima (px²) para ser considerado um carro
OFFSET          = 12    # tolerância em pixels ao cruzar a linha
COOLDOWN_FRAMES = 20    # frames de bloqueio após contar um carro
DIST_MINIMA     = 80    # distância máxima para associar ao mesmo carro
MAX_FRAMES      = 0     # 0 = processa tudo

# ─────────────────────────────────────────────────
# Inicialização
# ─────────────────────────────────────────────────
cap     = cv2.VideoCapture(ARQUIVO_VIDEO)
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps     = cap.get(cv2.CAP_PROP_FPS)

LINHA_Y = altura // 2

subtrator = cv2.createBackgroundSubtractorMOG2(
    history=300, varThreshold=60, detectShadows=True
)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

ARQUIVO_SAIDA = 'resultado_contagem.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
saida  = cv2.VideoWriter(ARQUIVO_SAIDA, fourcc, fps, (largura, altura))

contador_carros = 0
frame_idx       = 0
proximo_id      = 0
rastreados      = {}  # { id: {'cx': x, 'cy': y} }
contados        = {}  # { id: frame_em_que_foi_contado }

def encontrar_id_proximo(cx, cy, rastreados, dist_max=80):
    melhor_id, melhor_dist = None, dist_max
    for obj_id, pos in rastreados.items():
        dist = np.hypot(cx - pos['cx'], cy - pos['cy'])
        if dist < melhor_dist:
            melhor_dist, melhor_id = dist, obj_id
    return melhor_id

def cooldown_ativo(obj_id, frame_atual):
    return obj_id in contados and frame_atual - contados[obj_id] <= COOLDOWN_FRAMES

print('Processando vídeo...')

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if MAX_FRAMES > 0 and frame_idx >= MAX_FRAMES:
        break

    # ── 1. Subtração de fundo ──────────────────────────────────────
    mascara = subtrator.apply(frame)
    _, mascara = cv2.threshold(mascara, 200, 255, cv2.THRESH_BINARY)

    # ── 2. Limpeza morfológica ─────────────────────────────────────
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN,  kernel)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, kernel)
    mascara = cv2.dilate(mascara, kernel, iterations=2)

    # ── 3. Detecção de contornos ───────────────────────────────────
    contornos, _ = cv2.findContours(
        mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    # ── 4. Linha de contagem (amarela) ─────────────────────────────
    cv2.line(frame, (0, LINHA_Y), (largura, LINHA_Y), (0, 255, 255), 2)

    ids_atuais = {}

    for contorno in contornos:
        area = cv2.contourArea(contorno)
        if area < AREA_MINIMA:
            continue

        x, y, w, h = cv2.boundingRect(contorno)
        cx = x + w // 2
        cy = y + h // 2

        obj_id = encontrar_id_proximo(cx, cy, rastreados)
        if obj_id is None:
            obj_id = proximo_id
            proximo_id += 1
            rastreados[obj_id] = {'cx': cx, 'cy': cy}

        ids_atuais[obj_id] = {'cx': cx, 'cy': cy}

        cor = (0, 255, 0)
        cv2.rectangle(frame, (x, y), (x + w, y + h), cor, 2)
        cv2.circle(frame, (cx, cy), 6, cor, -1)
        cv2.putText(frame, f'ID {obj_id}', (x, y - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, cor, 2)

        # ── 5. Verifica cruzamento da linha ────────────────────────
        if (LINHA_Y - OFFSET) < cy < (LINHA_Y + OFFSET):
            if not cooldown_ativo(obj_id, frame_idx):
                contador_carros += 1
                contados[obj_id] = frame_idx
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 3)
                cv2.putText(frame, 'CONTADO', (x, y - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    rastreados = ids_atuais

    # ── 6. Placar ──────────────────────────────────────────────────
    cv2.rectangle(frame, (0, 0), (240, 55), (0, 0, 0), -1)
    cv2.putText(frame, f'Carros: {contador_carros}',
                (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (255, 255, 255), 3)

    saida.write(frame)
    frame_idx += 1

cap.release()
saida.release()

print(f'\n✅ Processamento concluído!')
print(f'   Frames processados : {frame_idx}')
print(f'   Carros contados    : {contador_carros}')
print(f'   Vídeo salvo em     : {ARQUIVO_SAIDA}')

## 07. Visualização dos Resultados

Exibe frames do vídeo processado para conferir visualmente a contagem.

In [ ]:
cap_resultado   = cv2.VideoCapture(ARQUIVO_SAIDA)
total_resultado = int(cap_resultado.get(cv2.CAP_PROP_FRAME_COUNT))

proporcoes = [0.1, 0.25, 0.4, 0.55, 0.7, 0.9]
posicoes   = [int(total_resultado * p) for p in proporcoes]

fig, eixos = plt.subplots(2, 3, figsize=(18, 8))
eixos = eixos.ravel()

for i, pos in enumerate(posicoes):
    cap_resultado.set(cv2.CAP_PROP_POS_FRAMES, pos)
    ret, frame = cap_resultado.read()
    if ret:
        eixos[i].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        eixos[i].set_title(f'Frame {pos}')
        eixos[i].axis('off')

cap_resultado.release()

plt.suptitle(f'Vídeo Processado — Total de Carros Contados: {contador_carros}', fontsize=14)
plt.tight_layout()
plt.show()

## 08. Comparação: Máscara Bruta vs Máscara Limpa

Mostra o efeito das operações morfológicas na qualidade da detecção.

In [ ]:
cap = cv2.VideoCapture(ARQUIVO_VIDEO)
subtrator2 = cv2.createBackgroundSubtractorMOG2(history=300, varThreshold=60, detectShadows=True)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

cap.set(cv2.CAP_PROP_POS_FRAMES, 150)
for _ in range(50):
    ret, frame = cap.read()
    if ret:
        subtrator2.apply(frame)

ret, frame = cap.read()
cap.release()

mascara_bruta = subtrator2.apply(frame)
_, mascara_bruta = cv2.threshold(mascara_bruta, 200, 255, cv2.THRESH_BINARY)
mascara_limpa = cv2.morphologyEx(mascara_bruta, cv2.MORPH_OPEN, kernel)
mascara_limpa = cv2.morphologyEx(mascara_limpa, cv2.MORPH_CLOSE, kernel)
mascara_limpa = cv2.dilate(mascara_limpa, kernel, iterations=2)

fig, eixos = plt.subplots(1, 3, figsize=(18, 5))
eixos[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); eixos[0].set_title('Frame Original');              eixos[0].axis('off')
eixos[1].imshow(mascara_bruta, cmap='gray');             eixos[1].set_title('Máscara Bruta (MOG2)');        eixos[1].axis('off')
eixos[2].imshow(mascara_limpa, cmap='gray');             eixos[2].set_title('Máscara Limpa (morfologia)'); eixos[2].axis('off')

plt.suptitle('Efeito do Pré-processamento na Máscara de Detecção', fontsize=13)
plt.tight_layout()
plt.show()

## 09. Histograma da Máscara

Análise da distribuição de pixels: fundo (0) vs movimento detectado (255).

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(mascara_limpa.ravel(), bins=10, color='steelblue', edgecolor='black')
plt.title('Histograma da Máscara — Pixels de Foreground vs Background')
plt.xlabel('Intensidade do Pixel (0 = fundo  |  255 = movimento)')
plt.ylabel('Quantidade de Pixels')
plt.xticks([0, 128, 255], ['0 (fundo)', '128', '255 (movimento)'])
plt.tight_layout()
plt.show()

pixels_movimento = int(np.sum(mascara_limpa == 255))
pixels_fundo     = int(np.sum(mascara_limpa == 0))
total_pixels     = mascara_limpa.size

print(f'Pixels de movimento : {pixels_movimento:,} ({100*pixels_movimento/total_pixels:.1f}%)')
print(f'Pixels de fundo     : {pixels_fundo:,} ({100*pixels_fundo/total_pixels:.1f}%)')

## 10. Resultado Final

In [ ]:
print('=' * 45)
print('        RESULTADO FINAL DO CONTADOR')
print('=' * 45)
print(f'  Vídeo processado  : {ARQUIVO_VIDEO}')
print(f'  Frames analisados : {frame_idx}')
print(f'  Carros contados   : {contador_carros}')
print(f'  Vídeo de saída    : {ARQUIVO_SAIDA}')
print('=' * 45)
print()
print('Técnica utilizada:')
print('  - Subtração de fundo : MOG2 (Mixture of Gaussians)')
print('  - Limpeza de ruído   : Morfologia (Open + Close + Dilate)')
print('  - Rastreamento       : Centroide por ID entre frames')
print('  - Contagem           : Linha horizontal com cooldown por ID')
print(f'  - Filtro de área     : Ignora objetos < {AREA_MINIMA} px²')

---
## Referências

- OpenCV Documentation: https://docs.opencv.org  
- Background Subtraction (MOG2): https://docs.opencv.org/4.x/d1/dc5/tutorial_background_subtraction.html  
- Cascade Classifier: https://docs.opencv.org/3.4/db/d28/tutorial_cascade_classifier.html  